# SED fitting

### Perform the fit with `Sedfitter` functions

##### Requirements: 
- Richardson grids filtered and convolved with Gaia (notebook 01)
- `Sedfitter` package from Richardson, T. (2024) available via [GitHub](https://github.com/richardson-t/sedfitter) 

### 

#### Step 1. Input file 

`Sedfitter` routine needs an input file with the data in a specific format defined in [Sedfitter docs | Data format](https://sedfitter.readthedocs.io/en/stable/data.html). 

In [ ]:
import os

os.chdir("../functions")
import input_file as inf
import fitter
import reading

The function `input_sedfitter` from `input.py` creates the input file from a csv with coordinates, photometric magnitudes and errors. Points without error are considered ad upper limits with 50% confidence. 

Zero points of the filters (in mJy) are avaiable in [SVO Filter Profile Service](https://svo2.cab.inta-csic.es/theory/fps3/).

In [ ]:
fin= '.csv'       # .csv file with photometric data in magnitudes
fout= '.txt' 

RA, DE = 'RAJ2000', 'DEJ2000'      # name of columns with coordinates 
bands = ['G','BP', 'RP', 'J', 'H', 'K','W1', 'W2', 'W3', 'W4']    # name of filters used, followed with 'mag' in dataframe

zero_points= [ 3228750, 3552010, 2554950,      # zero points of the filters, in the same order 
               1594000, 1024000, 666800,
               309540, 171790, 31670, 8360]

confidence= 0.5    # 50% confidence in measures without error

inf.input_sedfitter(fin, fout, RA, DE, bands, zero_points, confidence)

Once we have the input text file, we can initialize the rest of parameters to perform the fit.

###

#### Step 2. Fit

`out_format` indicates how many models to output. `N` means num_out is a fixed number, but it can be changed according to [Model selection syntaxis](https://sedfitter.readthedocs.io/en/stable/select_syntax.html). In our analysis, 1000 templates have been enough for all sources, but it may need more depending on the case. 

In [ ]:
folder_out= './out/'  # path of the output new directory

num_out= 1000                 
out_format= ('N',num_out)    # the output will retrieve 1000 templates


filters = ['GAIA/GAIA3.G','GAIA/GAIA3.Gbp','GAIA/GAIA3.Grp',                    # same order as before as named in Sedfitter family folders
           '2MASS/2MASS.J','2MASS/2MASS.H', '2MASS/2MASS.Ks', 
           'WISE/WISE.W1', 'WISE/WISE.W2', 'WISE/WISE.W3', 'WISE/WISE.W4']

families = ['s---s-i','sp--s-i','sp--h-i','s---smi','sp--smi','sp--hmi', 's-p-smi','s-p-hmi',
            's-pbsmi','s-pbhmi', 's-u-smi','s-u-hmi','s-ubsmi','s-ubhmi', 'spu-smi','spu-hmi','spubsmi', 'spubhmi'] 

fitter.fit_routine(fin, fout, folder_out, out_format, filters, families)

This routine performs the fit, retrieves `.fitinfo` files and a text file with the parameters of the 1000 best templates. Later it joins everything in `folder_out` in a directory with `result_+family name`. 

Be careful with the memory of your device. `Sedfitter` creates a bunch of temporary files and folders and in some cases they might occupy about a GB in disk per source. We have tried to optimize this the best we can but without changing anything in `Sedfitter` functions.

### 

#### Step 3. Reading the output 

Lastly, we read the fitinfo files, create a new csv file with the parameters of the best 1000 fits per family, and select the best models according to method *K*, method *F* and *BIC*. *K* and *F* thresholds can be chosen here. We also save a data frame with the considered good models of each method. Everything will be saved in `output_folder`

In [ ]:
F_thres= 3
K_thres= 9       # (as in R17)

reading.read_results(fin, folder_out, K_thres, F_thres, families)

Now we have the Sedfitter outputs in a more handy format and can select the best according to the preferred method, and visualize. We can continue to notebook `04_analysis.ipynb`.